# 18A — SHARP Temporal Development Baseline

## Purpose

First substantive SHARP-only temporal development baseline for the Cycle-24 → Cycle-25 extension.

This notebook **does not replace** the historical image-only, magnetic-only, or intermediate-fusion baseline experiments already preserved in the repository.

## Forecasting contract

Predict same-active-region **M/X flare occurrence in (t, t+48 h]**.

## Input representation

Three historical SHARP records:

- t − 288 min
- t − 192 min
- t − 96 min

with 15 magnetic features per step, flattened in temporal order to 45 classical-model inputs.

## Conservative source-quality rule

All three historical records must satisfy:

- QUALITY = 0
- all 15 available magnetic features finite
- exactly one source-record match
- no explicit NOAA identity conflict
- three unique exact historical records

No imputation is used in this first conservative reference experiment.

## Development protocol

Only the pre-specified Cycle-24 forward-development folds are used:

- cycle24_forward_validate_2013
- cycle24_forward_validate_2014
- cycle24_forward_validate_2015

Active-region connected components must be disjoint between training and validation.

The separately reserved Cycle-24 calibration holdout, threshold holdout, Cycle-25 independent test, and 2026 supplementary evaluation are not used here.

## Models

1. Logistic Regression with StandardScaler fit on each training fold only.
2. Random Forest.

## Threshold policy

Both threshold 0.5 and the TSS-maximising threshold selected on the same development validation fold are recorded.

The validation-selected threshold result is a **development diagnostic**, not an unbiased independent-test estimate.

## Metrics

ROC-AUC, PR-AUC, Brier score, TSS, HSS, precision, recall, F1, TN, FP, FN, TP.

## Scientific-status warning

At this stage, label clearance and historical SHARP availability/contributing-time support remain pending, and the source manifest still records `training_authorised = False`.

Therefore this notebook must not be described as a final scientifically cleared Cycle-24 → Cycle-25 forecasting result.


In [1]:
from pathlib import Path
import ast, gzip, json, time
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

BASE = Path.home() / "aia_sharp_cycle24_input" / "20260916T195026327454Z"
OUT = Path.home() / "sharp_dev_baseline_20260917"
PRED_DIR = OUT / "predictions"
MODEL_DIR = OUT / "models"
for p in (OUT, PRED_DIR, MODEL_DIR):
    p.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 20260917
print("BASE:", BASE)
print("OUT:", OUT)


BASE: /home/abmoses2000/aia_sharp_cycle24_input/20260916T195026327454Z
OUT: /home/abmoses2000/sharp_dev_baseline_20260917


## 1. Load exact saved Cycle-24 arrays and metadata

In [2]:
X3 = np.load(BASE / "sharp_raw_three_slot.npy", mmap_mode="r")
finite = np.load(BASE / "sharp_finite_mask.npy", mmap_mode="r")
matches = np.load(BASE / "sharp_record_match_counts.npy", mmap_mode="r")
y = np.load(BASE / "original_manifest_targets.npy", mmap_mode="r")

rows = pd.read_csv(BASE / "cycle24_array_rows.csv.gz")
folds = pd.read_csv(BASE / "forward_development_array_rows.csv.gz")

assert X3.shape == (64725, 3, 15)
assert finite.shape == X3.shape
assert matches.shape == (64725, 3)
assert y.shape == (64725,)
assert len(rows) == 64725

X45 = np.asarray(X3).reshape(len(X3), -1)
assert X45.shape == (64725, 45)

print("X temporal:", X3.shape)
print("X flattened:", X45.shape)
print("targets:", y.shape)
print("Cycle-24 rows:", rows.shape)
print("development assignments:", folds.shape)


X temporal: (64725, 3, 15)
X flattened: (64725, 45)
targets: (64725,)
Cycle-24 rows: (64725, 15)
development assignments: (104748, 8)


## 2. Reconstruct conservative source-quality eligibility

In [3]:
source_quality = {}
with gzip.open(BASE / "unique_sharp_source_records.jsonl.gz", "rt") as handle:
    for line in handle:
        rec = json.loads(line)
        idx = int(rec["slot_source_index"])
        raw_quality = rec["raw_QUALITY"]
        if isinstance(raw_quality, list):
            quality_zero = len(raw_quality) == 1 and str(raw_quality[0]).strip() == "0"
        else:
            quality_zero = str(raw_quality).strip() == "0"
        source_quality[idx] = {
            "quality_zero": quality_zero,
            "finite_feature_count": int(rec["finite_feature_count"]),
            "match_count": int(rec["match_count"]),
            "noaa_conflict": bool(rec["noaa_conflict"]),
        }

quality_pass = np.zeros(len(rows), dtype=bool)
for i, raw_indices in enumerate(rows["slot_source_indices"]):
    inds = ast.literal_eval(raw_indices)
    if len(inds) != 3:
        continue
    quality_pass[i] = all(
        (q := source_quality.get(int(idx))) is not None
        and q["quality_zero"]
        and q["finite_feature_count"] == 15
        and q["match_count"] == 1
        and not q["noaa_conflict"]
        for idx in inds
    )

integrity_pass = (
    finite.all(axis=(1, 2))
    & (matches == 1).all(axis=1)
    & (rows["three_unique_exact_records"].to_numpy() == 1)
    & (rows["raw_numeric_complete"].to_numpy() == 1)
)
eligible = quality_pass & integrity_pass

print("All Cycle-24 candidates:", len(rows))
print("Conservative eligible:", int(eligible.sum()))
print("Excluded:", int((~eligible).sum()))
print("Eligible positives:", int(y[eligible].sum()))


All Cycle-24 candidates: 64725
Conservative eligible: 55871
Excluded: 8854
Eligible positives: 1800


## 3. Verify development-only contract and region separation

In [4]:
row_meta = rows[
    ["array_row", "stored_year", "HARPNUM", "NOAA_AR_clean",
     "region_component_id", "proposed_final_role", "original_label_48h_final"]
].copy()
row_meta["eligible"] = eligible
row_meta["array_label"] = y

dev = folds.merge(
    row_meta,
    on=["array_row", "region_component_id"],
    how="left",
    validate="many_to_one",
)

roles_used = set(dev["proposed_final_role"].dropna().unique())
print("Final-role classes present in development folds:", sorted(roles_used))
if roles_used != {"cycle24_final_refit_pool"}:
    raise RuntimeError("Development folds contain rows outside cycle24_final_refit_pool.")

if not np.array_equal(
    dev["original_label_48h_final"].to_numpy(),
    dev["array_label"].to_numpy()
):
    raise RuntimeError("Row labels disagree with saved target array.")

if (dev["stored_year"] >= 2020).any():
    raise RuntimeError("Cycle-25-era observation reached development.")

usable = dev[(dev["structural_candidate"] == 1) & (dev["eligible"])].copy()

for fold_id, d in usable.groupby("fold_id"):
    tr = set(d.loc[d["fold_role"] == "train", "region_component_id"])
    va = set(d.loc[d["fold_role"] == "validation", "region_component_id"])
    overlap = tr & va
    print(f"{fold_id}: train={sum(d.fold_role=='train')}, validation={sum(d.fold_role=='validation')}, overlap_regions={len(overlap)}")
    if overlap:
        raise RuntimeError(f"Region leakage detected in {fold_id}")

print("STATUS: DEVELOPMENT_CONTRACT_VERIFIED")


Final-role classes present in development folds: ['cycle24_final_refit_pool']
cycle24_forward_validate_2013: train=15203, validation=7586, overlap_regions=0
cycle24_forward_validate_2014: train=22963, validation=7102, overlap_regions=0
cycle24_forward_validate_2015: train=30187, validation=6649, overlap_regions=0
STATUS: DEVELOPMENT_CONTRACT_VERIFIED


## 4. Metrics and threshold helpers

In [5]:
def tss_from_cm(tn, fp, fn, tp):
    tpr = tp / (tp + fn) if (tp + fn) else np.nan
    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    return tpr - fpr

def hss_from_cm(tn, fp, fn, tp):
    num = 2 * (tp * tn - fn * fp)
    den = (tp + fn) * (fn + tn) + (tp + fp) * (fp + tn)
    return num / den if den else np.nan

def optimal_tss_threshold(y_true, prob):
    fpr, tpr, thresholds = roc_curve(y_true, prob)
    score = tpr - fpr
    finite_idx = np.where(np.isfinite(thresholds))[0]
    idx = finite_idx[np.argmax(score[finite_idx])]
    return float(thresholds[idx]), float(score[idx])

def classification_metrics(y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "tss": float(tss_from_cm(tn, fp, fn, tp)),
        "hss": float(hss_from_cm(tn, fp, fn, tp)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
    }


## 5. Define prespecified baseline models

In [6]:
models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            class_weight="balanced",
            max_iter=5000,
            solver="lbfgs",
            random_state=RANDOM_STATE,
        )),
    ]),
    "random_forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced_subsample",
        max_features="sqrt",
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}
models


{'logistic_regression': Pipeline(steps=[('scale', StandardScaler()),
                 ('model',
                  LogisticRegression(class_weight='balanced', max_iter=5000,
                                     random_state=20260917))]),
 'random_forest': RandomForestClassifier(class_weight='balanced_subsample', min_samples_leaf=2,
                        n_estimators=500, n_jobs=-1, random_state=20260917)}

## 6. Train only on pre-specified Cycle-24 development folds

In [7]:
results = []
for fold_id in sorted(usable["fold_id"].unique()):
    fold = usable[usable["fold_id"] == fold_id].copy()
    train = fold[fold["fold_role"] == "train"].copy()
    val = fold[fold["fold_role"] == "validation"].copy()

    train_idx = train["array_row"].to_numpy(dtype=int)
    val_idx = val["array_row"].to_numpy(dtype=int)

    X_train, X_val = X45[train_idx], X45[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    if not np.isfinite(X_train).all() or not np.isfinite(X_val).all():
        raise RuntimeError(f"Nonfinite inputs after conservative filter in {fold_id}")

    print("\nFOLD:", fold_id)
    print("train:", len(train_idx), "positives:", int(y_train.sum()), "regions:", train["region_component_id"].nunique())
    print("validation:", len(val_idx), "positives:", int(y_val.sum()), "regions:", val["region_component_id"].nunique())

    for model_name, estimator in models.items():
        t0 = time.time()
        estimator.fit(X_train, y_train)
        fit_seconds = time.time() - t0

        val_prob = estimator.predict_proba(X_val)[:, 1]
        roc_auc = roc_auc_score(y_val, val_prob)
        pr_auc = average_precision_score(y_val, val_prob)
        brier = brier_score_loss(y_val, val_prob)

        fixed = classification_metrics(y_val, val_prob, 0.5)
        best_threshold, _ = optimal_tss_threshold(y_val, val_prob)
        tuned = classification_metrics(y_val, val_prob, best_threshold)

        results.append({
            "fold_id": fold_id,
            "model": model_name,
            "n_train": int(len(train_idx)),
            "train_positive": int(y_train.sum()),
            "n_validation": int(len(val_idx)),
            "validation_positive": int(y_val.sum()),
            "train_regions": int(train["region_component_id"].nunique()),
            "validation_regions": int(val["region_component_id"].nunique()),
            "fit_seconds": float(fit_seconds),
            "roc_auc": float(roc_auc),
            "pr_auc": float(pr_auc),
            "brier": float(brier),
            "fixed_0.5": fixed,
            "validation_tss_optimised": tuned,
            "threshold_note": "Selected and evaluated on the same development validation fold; not an independent performance estimate.",
        })

        pred = val[["target_sample_id", "array_row", "region_component_id", "stored_year"]].copy()
        pred["y_true"] = y_val
        pred["probability"] = val_prob
        pred["pred_fixed_0_5"] = (val_prob >= 0.5).astype(int)
        pred["validation_selected_threshold"] = best_threshold
        pred["pred_validation_selected"] = (val_prob >= best_threshold).astype(int)
        pred.to_csv(PRED_DIR / f"{fold_id}__{model_name}.csv.gz", index=False, compression="gzip")
        joblib.dump(estimator, MODEL_DIR / f"{fold_id}__{model_name}.joblib")

        print(f"{model_name}: ROC-AUC={roc_auc:.4f} PR-AUC={pr_auc:.4f} Brier={brier:.4f} "
              f"TSS@0.5={fixed['tss']:.4f} TSS@val-opt={tuned['tss']:.4f}")



FOLD: cycle24_forward_validate_2013
train: 15203 positives: 383 regions: 337
validation: 7586 positives: 291 regions: 169


logistic_regression: ROC-AUC=0.9297 PR-AUC=0.4540 Brier=0.0902 TSS@0.5=0.6930 TSS@val-opt=0.7155


random_forest: ROC-AUC=0.9351 PR-AUC=0.4854 Brier=0.0266 TSS@0.5=0.1937 TSS@val-opt=0.7574

FOLD: cycle24_forward_validate_2014
train: 22963 positives: 674 regions: 510
validation: 7102 positives: 399 regions: 146


logistic_regression: ROC-AUC=0.9231 PR-AUC=0.4703 Brier=0.0946 TSS@0.5=0.7100 TSS@val-opt=0.7376


random_forest: ROC-AUC=0.8385 PR-AUC=0.3356 Brier=0.0457 TSS@0.5=0.1357 TSS@val-opt=0.5274

FOLD: cycle24_forward_validate_2015
train: 30187 positives: 1096 regions: 657
validation: 6649 positives: 347 regions: 138


logistic_regression: ROC-AUC=0.9549 PR-AUC=0.6189 Brier=0.0742 TSS@0.5=0.8163 TSS@val-opt=0.8176


random_forest: ROC-AUC=0.9334 PR-AUC=0.4271 Brier=0.0390 TSS@0.5=0.1051 TSS@val-opt=0.7662


## 7. Save compact results and protocol record

In [8]:
with open(OUT / "development_results.json", "w") as f:
    json.dump(results, f, indent=2)

flat = []
for r in results:
    flat.append({
        "fold_id": r["fold_id"], "model": r["model"],
        "n_train": r["n_train"], "train_positive": r["train_positive"],
        "n_validation": r["n_validation"], "validation_positive": r["validation_positive"],
        "roc_auc": r["roc_auc"], "pr_auc": r["pr_auc"], "brier": r["brier"],
        "fixed_threshold": r["fixed_0.5"]["threshold"],
        "fixed_tss": r["fixed_0.5"]["tss"],
        "fixed_hss": r["fixed_0.5"]["hss"],
        "fixed_precision": r["fixed_0.5"]["precision"],
        "fixed_recall": r["fixed_0.5"]["recall"],
        "fixed_f1": r["fixed_0.5"]["f1"],
        "selected_threshold": r["validation_tss_optimised"]["threshold"],
        "selected_tss": r["validation_tss_optimised"]["tss"],
        "selected_hss": r["validation_tss_optimised"]["hss"],
        "selected_precision": r["validation_tss_optimised"]["precision"],
        "selected_recall": r["validation_tss_optimised"]["recall"],
        "selected_f1": r["validation_tss_optimised"]["f1"],
        "fit_seconds": r["fit_seconds"],
    })

summary = pd.DataFrame(flat)
summary.to_csv(OUT / "development_results.csv", index=False)

protocol = {
    "status": "DEVELOPMENT_DIAGNOSTIC_COMPLETE_NOT_FINAL_TEST",
    "input_shape": [64725, 3, 15],
    "flattened_feature_count": 45,
    "history_minutes": [-288, -192, -96],
    "quality_rule": "All three source records require QUALITY=0, 15/15 finite features, exact single source match, no explicit NOAA conflict.",
    "models": ["logistic_regression", "random_forest"],
    "development_folds": sorted(usable["fold_id"].unique().tolist()),
    "cycle25_used": False,
    "final_calibration_holdout_used": False,
    "final_threshold_holdout_used": False,
    "final_test_used": False,
    "scientific_clearance": False,
    "reason_not_final": [
        "label clearance remains pending",
        "training_authorised remains false in source manifest",
        "SHARP historical availability/contributing-time review remains pending",
        "validation-selected threshold is evaluated on the same development validation fold",
    ],
}
with open(OUT / "protocol_record.json", "w") as f:
    json.dump(protocol, f, indent=2)

print(summary.to_string(index=False))
print("\nOUTPUT:", OUT)
print("STATUS: SHARP_DEVELOPMENT_BASELINE_COMPLETE_NO_CYCLE25_TEST_USED")


                      fold_id               model  n_train  train_positive  n_validation  validation_positive  roc_auc   pr_auc    brier  fixed_threshold  fixed_tss  fixed_hss  fixed_precision  fixed_recall  fixed_f1  selected_threshold  selected_tss  selected_hss  selected_precision  selected_recall  selected_f1  fit_seconds
cycle24_forward_validate_2013 logistic_regression    15203             383          7586                  291 0.929672 0.453951 0.090180              0.5   0.692961   0.303034         0.219421      0.807560  0.345081            0.379199      0.715541      0.231556            0.166881         0.893471     0.281233     1.782060
cycle24_forward_validate_2013       random_forest    15203             383          7586                  291 0.935114 0.485449 0.026594              0.5   0.193683   0.302454         0.780822      0.195876  0.313187            0.011856      0.757364      0.231263            0.165269         0.948454     0.281489     5.141547
cycle24_forward_

## 8. Interpretation checklist

After execution, record:

- fold-by-fold class support after quality filtering;
- whether Logistic Regression converged;
- threshold-0.5 and validation-selected TSS/HSS;
- ROC-AUC and PR-AUC;
- Brier score;
- any fold instability;
- whether conclusions differ materially between the two baseline models;
- all limitations above.

Do **not** claim Cycle-25 generalisation from this notebook. The independent Cycle-25 test remains untouched.
